<a href="https://colab.research.google.com/github/aishanikar9/BWSI_Operations_Team/blob/main/BWSI_xView2_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import pandas as pd
from skimage import io
import numpy as np
import matplotlib.pyplot as plt
import pathlib


In [ ]:
!pip install kagglehub

import kagglehub

# Download latest version
path = kagglehub.dataset_download("tunguz/xview2-challenge-dataset-train-and-test")
print("Path to dataset files:", path)

import os

# View all files and folders in the downloaded path
files = os.listdir(path)
print(files)


In [ ]:
# To inspect if folders are structures or archives, add this block:
for f in files:
    full_p = os.path.join(path, f)
    if os.path.isdir(full_p):
        print(f"Folder: {f} -> Contains: {os.listdir(full_p)}")
    else:
        print(f"File: {f}")


In [ ]:
# Check the contents inside the nested folders
print("Inside train/train:", os.listdir(os.path.join(path, 'train', 'train')))
print("Inside test/test:", os.listdir(os.path.join(path, 'test', 'test')))


In [5]:
!pip install shapely

first we find the bounding box of each building and crop them out into separate chips to make our new target dataset.

In [ ]:
import os
import json
import cv2
import pandas as pd
import numpy as np
import kagglehub
from shapely.wkt import loads
from sklearn.model_selection import train_test_split
from tqdm import tqdm

path = kagglehub.dataset_download("tunguz/xview2-challenge-dataset-train-and-test")

train_dir = os.path.join(path, 'train', 'train')
train_image_dir = os.path.join(train_dir, 'images')
train_label_dir = os.path.join(train_dir, 'labels')

test_dir = os.path.join(path, 'test', 'test')
test_image_dir = os.path.join(test_dir, 'images')

output_crop_dir = '/content/cropped_building_chips'
os.makedirs(output_crop_dir, exist_ok=True)

damage_map = {'no-damage': 0, 'minor-damage': 1, 'major-damage': 2, 'destroyed': 3}
post_jsons = [f for f in os.listdir(train_label_dir) if f.endswith('_post_disaster.json')]

all_train_val_chips = []
crop_counter = 0

print("--- Step 1: Extracting Building Chips from Training Images ---")
for j_file in tqdm(post_jsons):
    img_file = j_file.replace('.json', '.png')
    img_path = os.path.join(train_image_dir, img_file)

    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    with open(os.path.join(train_label_dir, j_file)) as f:
        label_data = json.load(f)

    for feature in label_data['features']['xy']:
        wkt_string = feature['wkt']
        damage_type = feature['properties']['subtype']

        if damage_type not in damage_map:
            continue

        try:
            poly = loads(wkt_string)
            xmin, ymin, xmax, ymax = poly.bounds

            xmin, ymin = max(0, int(xmin)), max(0, int(ymin))
            xmax, ymax = min(img.shape[1], int(xmax)), min(img.shape[0], int(ymax))

            crop = img[ymin:ymax, xmin:xmax]
            if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
                continue

            crop_filename = f"building_{crop_counter}.png"
            crop_path = os.path.join(output_crop_dir, crop_filename)
            cv2.imwrite(crop_path, crop)

            all_train_val_chips.append({
                'chip_path': crop_path,
                'damage_label': damage_map[damage_type]
            })
            crop_counter += 1

        except Exception:
            continue

print(f"\nExtracted {crop_counter} building chips.")

print("\n--- Step 2: Creating xView2 Train and Val CSV Catalogs ---")
df_all = pd.DataFrame(all_train_val_chips)

df_train, df_val = train_test_split(df_all, test_size=0.2, stratify=df_all['damage_label'], random_state=42)

df_train.to_csv('/content/xview2-classification-train.csv', index=False)
df_val.to_csv('/content/xview2-classification-val.csv', index=False)

print(f"Saved: xview2-classification-train.csv ({len(df_train)} records)")
print(f"Saved: xview2-classification-val.csv ({len(df_val)} records)")

print("\n--- Step 3: Cataloging Unlabeled xView2 Test Images ---")
test_images = [os.path.join(test_image_dir, f) for f in os.listdir(test_image_dir) if f.endswith('.png')]

df_test = pd.DataFrame({'image_path': test_images})
df_test.to_csv('/content/xview2-classification-test.csv', index=False)
print(f"Saved: xview2-classification-test.csv ({len(df_test)} raw testing images)")


Using Colab cache for faster access to the 'xview2-challenge-dataset-train-and-test' dataset.
--- Step 1: Extracting Building Chips from Training Images ---


  6%|▋         | 178/2799 [00:21<02:50, 15.39it/s]

In [ ]:
from google.colab import files

files.download('/content/xview2-classification-train.csv')
files.download('/content/xview2-classification-val.csv')
files.download('/content/xview2-classification-test.csv')

In [ ]:
#weighting each class

train_df = pd.read_csv('/content/xview2-classification-train.csv')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active training device: {device}")

# Re-weighting each class based on the number of examples
no_damage_weight = (len(train_df)-(train_df['damage_label']==0).sum())/(train_df['damage_label']==0).sum()
minor_damage_weight = (len(train_df)-(train_df['damage_label']==1).sum())/(train_df['damage_label']==1).sum()
major_damage_weight = (len(train_df)-(train_df['damage_label']==2).sum())/(train_df['damage_label']==2).sum()
destroyed_weight = (len(train_df)-(train_df['damage_label']==3).sum())/(train_df['damage_label']==3).sum()

class_weights = torch.as_tensor([no_damage_weight, minor_damage_weight, major_damage_weight, destroyed_weight], dtype=torch.float).to(device)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms

#Define Individual Transformation Components
flip        = transforms.RandomHorizontalFlip(p=0.5)
v_flip      = transforms.RandomVerticalFlip(p=0.5)
scale       = transforms.Resize((128, 128)) # Resizing step handled uniformly
rotation    = transforms.RandomRotation(degrees=30)
jitter      = transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15)
perspective = transforms.RandomPerspective(distortion_scale=0.2, p=0.4)

#Compose the Pipelines
#Training pipeline implements diverse augmentations to prevent noise overfitting
composed_train = transforms.Compose([
    scale,
    flip,
    v_flip,
    rotation,
    perspective,
    jitter,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Validation pipeline keeps images clean and unwarped for true performance tracking
composed_val = transforms.Compose([
    scale,
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#Create the Custom Dataset Class
class XView2Dataset(Dataset):
    def __init__(self, label_csv, transform=None):
        self.label_data_df = pd.read_csv(label_csv)
        self.transform = transform

    def __len__(self):
        return len(self.label_data_df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.label_data_df.iloc[idx]['chip_path']
        label = int(self.label_data_df.iloc[idx]['damage_label'])

        # Read the building crop image safely
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        example = {
            'image': image,
            'label': label
        }
        return example

#Instantiate Datasets
transformed_train_dataset = XView2Dataset('/content/xview2-classification-train.csv', transform=composed_train)
transformed_val_dataset   = XView2Dataset('/content/xview2-classification-val.csv', transform=composed_val)

#Set Up Parallel DataLoaders
train_loader = DataLoader(transformed_train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(transformed_val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision

#Load Pre-trained ResNet50 Backbone
torch.backends.cudnn.benchmark = True  # Flag for specialized GPU hardware optimizations
net = torchvision.models.get_model('resnet50', pretrained=True)

#Modify Final Classification Layer
# ResNet50 produces 2048 high-level feature extractions right before its output.
# Swaps the default layer with a fresh linear layer targeting the 4 damage categories (0 to 3).
net.fc = nn.Linear(2048, 4)

# Push the configured network architecture onto active GPU runtime device
net = net.to(device)
print("ResNet50 Backbone successfully anchored to hardware device and ready.")

criterion = nn.CrossEntropyLoss(weight=class_weights) #loss function
optimizer = optim.Adam(net.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [ ]:
#import pathlib
from torch.utils.tensorboard import SummaryWriter

def train_model(net, train_loader, val_loader, criterion, optimizer, scheduler,
                 logs_path, model_name, starting_epoch=0, additional_epochs=5,
                 print_every_num_batches=100):

    model_name_base = f'resnet50-{model_name}' + '.ep{}.pth'
    writer = SummaryWriter(logs_path)
    checkpoints_path = logs_path/'checkpoints'
    checkpoints_path.mkdir(parents=True, exist_ok=True)

    if starting_epoch > 0:
        starting_epoch_string = str(starting_epoch).zfill(3)
        model_load_path = checkpoints_path/model_name_base.format(starting_epoch_string)
        net.load_state_dict(torch.load(model_load_path))

    for epoch in range(starting_epoch, starting_epoch + additional_epochs):

        #training
        net.train()
        running_loss = 0.0
        running_epoch_loss = 0.0

        for i, data in enumerate(train_loader, 0):
            inputs = data['image'].to(device)
            labels = data['label'].to(device)

            optimizer.zero_grad()

            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            running_epoch_loss += loss.item()
            if (i+1) % print_every_num_batches == 0:  # print every N mini-batches
                print(f'[epoch {epoch+1}, batch {i+1}] average loss: {running_loss/print_every_num_batches}')
                running_loss = 0.0

        average_epoch_loss = running_epoch_loss/(i+1)
        writer.add_scalar('Loss/epoch_avg/train', average_epoch_loss, epoch)
        print(f'[epoch {epoch+1}] average training epoch loss: {average_epoch_loss}')

        writer.add_scalar('LR/rate', scheduler.get_last_lr()[0], epoch)
        scheduler.step()

        #Validation:
        net.eval()
        running_epoch_loss = 0.0
        correct = 0
        total = 0
        print("Getting epoch validation loss...")
        with torch.no_grad():
            for i, data in enumerate(val_loader, 0):
                inputs = data['image'].to(device)
                labels = data['label'].to(device)

                outputs = net(inputs)
                loss = criterion(outputs, labels)
                running_epoch_loss += loss.item()

                # single label prediction, pick the highest-scoring class
                predicted = outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        average_epoch_loss = running_epoch_loss/(i+1)
        val_accuracy = 100 * correct / total
        writer.add_scalar('Loss/epoch_avg/val', average_epoch_loss, epoch)
        writer.add_scalar('Accuracy/epoch_avg/val', val_accuracy, epoch)
        print(f'[epoch {epoch+1}] average val epoch loss: {average_epoch_loss}')
        print(f'[epoch {epoch+1}] val accuracy: {val_accuracy:.2f}%')

        epoch_string = str(epoch+1).zfill(3)
        model_save_path = checkpoints_path/model_name_base.format(epoch_string)
        torch.save(net.state_dict(), model_save_path)

    print('Finished Training')
    writer.close()

In [ ]:
outputs = pathlib.Path('outputs')
outputs.mkdir(exist_ok=True, parents=True)

model_name = 'damage_model'

# start with 5 epochs
train_model(net, train_loader, val_loader, criterion, optimizer, scheduler, outputs, model_name,
            starting_epoch=0, additional_epochs=5, print_every_num_batches=100)